In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(f"{path}/Q1_data.csv")


In [ ]:
# Extra print the Dataset shape
print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

# Delivery Time distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head()

In [ ]:
# drop the rows that have missing our TARGET
df = df.dropna(subset=['Delivery_Time'])

In [ ]:
print(f"Dataset shape: {df.shape}")
df.isnull().sum()

In [ ]:
# for the Categorical Features: Weather - Traffic_Level - Time_of_Day fill with mode
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])

In [ ]:
# for the Numeric Feature: Courier_Experience_yrs fill with the mean
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

In [ ]:
print(f"Dataset shape: {df.shape}")
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))



In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:

# The target is continuous there is nothing called target imbalance :)

In [ ]:
# Task 1: Write your code here:

X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

In [ ]:
# Task 2,3,4,5: Write your code here:

import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store MAE scores for each fold
mae_scores = []

# Store the last model for feature importance later
final_model = None

# Store predictions for plotting later
all_predictions = []
all_actuals = []

print("Starting 5-Fold Cross-Validation with RandomForest...")
print("=" * 55)

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"\nFold {fold}/{5}")

    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train RandomForest model
    model = RandomForestRegressor(
        n_estimators=100,    # Number of trees
        max_depth=10,        # Max depth of trees
        random_state=42,
        n_jobs=-1            # Use all CPU cores
    )
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_val)

    # MAE
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

    print(f"  MAE: {mae:.4f}")

    # Store for later
    final_model = model
    all_predictions.extend(y_pred)
    all_actuals.extend(y_val)

# Print averaged score
print("\n" + "=" * 55)
print("CROSS-VALIDATION RESULTS")
print("=" * 55)
print(f"\nAverage MAE: {np.mean(mae_scores):.4f} ")

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.median())

baseline_mse = mean_absolute_error(y, baseline_pred)

print(f"Baseline MAE (Mean Absolute Error): {baseline_mse:.4f}")

In [ ]:
# Task 1: Write your code here:

# RandomForest provides feature_importances_ attribute

# Get feature importances
importances = final_model.feature_importances_
feature_names = X.columns

# Create DataFrame and sort
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_n = min(15, len(importance_df))
top_features = importance_df.head(top_n)

plt.barh(range(top_n), top_features['Importance'].values[::-1], color='steelblue')
plt.yticks(range(top_n), top_features['Feature'].values[::-1])
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.title('Top Feature Importances for Delivery Time Prediction')
plt.tight_layout()
plt.show()

# Print top 5
print("\nTop 5 Most Important Features:")
print(importance_df.head())

In [ ]:
# Task 2: Write your code here:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of predictons vs actuals
axes[0].hist(all_actuals, bins=30, alpha=0.7, label='Actual', color='blue', edgecolor='black')
axes[0].hist(all_predictions, bins=30, alpha=0.7, label='Predicted', color='orange', edgecolor='black')
axes[0].set_xlabel('Delivery Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Actual vs Predicted Delivery Time Distribution')
axes[0].legend()

#  scatter plot of actual vs predicted
axes[1].scatter(all_actuals, all_predictions, alpha=0.5, edgecolors='black', linewidth=0.5)
min_val = min(min(all_actuals), min(all_predictions))
max_val = max(max(all_actuals), max(all_predictions))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Delivery Time (minutes)')
axes[1].set_ylabel('Predicted Delivery Time (minutes)')
axes[1].set_title('Actual vs Predicted Delivery Time')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:
